# KPI Interpretation Tool: Automating Four-Layer Analytics Narrative with the Claude API
**Author:** Adam Zavala
**Date:** May 2026
**Domain:** Applied GenAI | LLM Integration | Analytics Automation
**Estimated Read Time:** 15 minutes

---

## Business Context

Analytics functions in large service organizations spend a disproportionate share of analyst time on the interpretive layer of KPI review -- the work that happens between "pull the metric" and "tell the stakeholder what it means and what to do next." For a team of four analysts supporting 30+ service managers across weekly operational reviews, this layer consumes hours that could be redirected toward causal analysis, measurement design, or model development.

This notebook demonstrates a working prototype of a KPI interpretation tool using the Claude API. It takes a metric time series and returns a structured four-layer interpretation: what happened, a signal vs. noise verdict, ordered candidate explanations, and a specific recommended action with a named threshold. The tool is designed to automate the first interpretive pass -- the layer that currently requires an analyst to read, contextualize, and summarize each metric before surfacing findings to service managers.

## Analytical Question

Can a structured system prompt reliably produce decision-ready KPI interpretations that distinguish between trends, discrete anomalies, and noise -- across different signal types -- without post-hoc manual correction?

## Key Finding

Yes, with deliberate prompt engineering. The system prompt enforces a four-field JSON structure with specific analytical requirements per field: the signal vs. noise verdict must conclude with a categorical classification (STRONG SIGNAL / MODERATE SIGNAL / NOISE), candidate explanations must be ordered by likelihood and name specific mechanisms rather than categories, and recommended actions must include a named threshold and decision trigger.

Tested on three IT service operations KPIs with distinct signal patterns -- a strong 12-week upward trend, a mid-series spike followed by recovery, and a late-period deterioration in a stable series -- the tool produces defensible output on all three on the first prompt attempt. The hardest test (late deterioration detection) passes because the user message pre-computes baseline average and recent average, allowing the model to compare against a stable reference rather than the full-series mean.

## Methodology Overview

The tool uses the Anthropic Python SDK to call `claude-sonnet-4-6` with a structured system prompt that enforces JSON-only output. The user message pre-computes four summary statistics (baseline avg, recent avg, overall % change, and the weekly time series) before passing to the model -- reducing arithmetic burden and improving quantitative consistency in the output. The signal verdict is enforced in the system prompt as a required classification, not inferred post-hoc. A code fence stripping step handles model version inconsistency on JSON wrapping. Output is rendered in two formats: a formatted terminal report with signal badges and a GitHub-renderable markdown export.

**Why this design over alternatives:** A simpler prompt asking "summarize this KPI" would produce description, not interpretation. The four-layer structure forces the model to commit to a signal assessment, generate ordered causal hypotheses, and produce a specific action -- the three things that distinguish analytical interpretation from narrative summary. Pre-computing summary statistics rather than asking the model to compute them reduces numeric hallucination risk and ensures consistent quantitative references across metrics.


In [1]:
# ============================================================================
# KPI INTERPRETATION TOOL
# Using the Claude API to automate four-layer KPI narrative generation
# Portfolio Artifact 1 | Director-Track Technical Depth Track | May 2026
# ============================================================================
#
# Four-layer interpretation framework:
#   Layer 1 — What happened (trend: direction, magnitude, duration)
#   Layer 2 — Signal vs. noise (is the change meaningful?)
#   Layer 3 — Candidate explanations (plausible causes)
#   Layer 4 — Recommended action (what should a decision-maker do?)
#
# This structure maps directly to the Analytics Leadership Philosophy
# four-question north star: what happened / why / what's next / what to do.
# The tool operationalizes that philosophy using the Claude API.
# ============================================================================

from dotenv import load_dotenv
import os
import anthropic
import pandas as pd
import json

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
print("Client initialized. Model: claude-sonnet-4-6")

Client initialized. Model: claude-sonnet-4-6


In [2]:
# ============================================================================
# SYNTHETIC KPI DATASET
# Mimics IT service operations metrics — same domain as the chatbot and
# knowledge tool studies. Three KPIs across 12 weeks.
# ============================================================================

import pandas as pd
import numpy as np

np.random.seed(42)

dates = pd.date_range(start="2026-01-05", periods=12, freq="W-MON")

kpi_data = pd.DataFrame([
    # Zero Touch Rate — strong upward trend with noise
    *[{"metric_name": "Zero Touch Rate",
       "date": d,
       "value": round(0.055 + i * 0.018 + np.random.normal(0, 0.008), 3),
       "unit": "rate",
       "context": "% of support interactions resolved without agent involvement"}
      for i, d in enumerate(dates)],

    # Ticket Volume — spike in weeks 4-6 then recovery
    *[{"metric_name": "Weekly Ticket Volume",
       "date": d,
       "value": int(np.random.normal(
           420 if i < 3 else (580 if 3 <= i <= 5 else 430), 25)),
       "unit": "count",
       "context": "Total IT support tickets submitted per week"}
      for i, d in enumerate(dates)],

    # Resolution Rate — stable with slight late dip
    *[{"metric_name": "Closed Within 1 Weekday Rate",
       "date": d,
       "value": round(np.random.normal(
           0.71 if i < 9 else 0.64, 0.02), 3),
       "unit": "rate",
       "context": "% of tickets resolved within one business day"}
      for i, d in enumerate(dates)],
])

kpi_data["date"] = pd.to_datetime(kpi_data["date"])

print(f"Dataset shape: {kpi_data.shape}")
print(f"Metrics: {kpi_data['metric_name'].unique().tolist()}")
print(f"Date range: {kpi_data['date'].min().date()} to {kpi_data['date'].max().date()}")
print()
print(kpi_data.groupby("metric_name").agg(
    weeks=("date", "count"),
    min_val=("value", "min"),
    max_val=("value", "max"),
    mean_val=("value", "mean")
).round(3).to_string())

Dataset shape: (36, 5)
Metrics: ['Zero Touch Rate', 'Weekly Ticket Volume', 'Closed Within 1 Weekday Rate']
Date range: 2026-01-05 to 2026-03-23

                              weeks  min_val  max_val  mean_val
metric_name                                                    
Closed Within 1 Weekday Rate     12    0.616    0.747     0.689
Weekly Ticket Volume             12  372.000  587.000   449.667
Zero Touch Rate                  12    0.059    0.249     0.156


---

## Prompt Engineering Design Decisions

The quality of the output depends entirely on the system prompt design. Four decisions are worth documenting explicitly before reviewing the implementation:

**1. Pre-computed summary statistics in the user message**
The user message passes baseline average, recent average, and percentage change before asking for interpretation. This reduces the model's arithmetic burden and produces more consistent quantitative references -- the model cites numbers you computed rather than deriving its own, which reduces hallucination risk on numerical claims.

**2. Signal verdict enforced in the system prompt, not inferred post-hoc**
The signal classification (STRONG SIGNAL / MODERATE SIGNAL / NOISE) is required as part of the `signal_vs_noise` field. This means the model commits to a classification as part of its reasoning, not as a label applied afterward. The `format_signal_badge()` function extracts the verdict and maps it to an emoji badge for display.

**3. JSON-only output with no preamble**
The system prompt instructs the model to return only valid JSON with exactly four fields. A code fence stripping step handles cases where the model wraps JSON in markdown backticks -- an inconsistency across model versions that would otherwise cause parsing failures.

**4. Ordered candidate explanations**
The prompt specifies "ordered from most to least likely" for candidate explanations. Without this instruction, models tend to list mechanisms in arbitrary order. Ordering forces a likelihood judgment rather than list-dumping, which is the analytical behavior the tool is designed to surface.


In [3]:
# ============================================================================
# CELL 3: SYSTEM PROMPT + SINGLE KPI INTERPRETATION
# Start with one KPI (Zero Touch Rate) to assess output quality
# before building the full loop.
# ============================================================================

SYSTEM_PROMPT = """You are a senior analytics interpreter embedded in an IT 
service operations team. Your job is to read KPI time series data and produce 
structured, decision-ready interpretations for senior operations leaders.

For each KPI you receive, produce a JSON object with exactly these four fields:

{
  "what_happened": "2-3 sentences describing the trend in plain language — 
    direction, magnitude, and duration. Name specific numbers. Do not use 
    vague language like 'the metric changed'.",

  "signal_vs_noise": "1-2 sentences assessing whether the observed change is 
    meaningful. Reference the magnitude of change relative to the baseline 
    and any volatility visible in the series. Conclude with a clear signal 
    verdict: STRONG SIGNAL, MODERATE SIGNAL, or NOISE.",

  "candidate_explanations": "2-4 bullet points listing the most plausible 
    explanations for the observed pattern, ordered from most to least likely. 
    Each bullet should name a specific mechanism, not just a category.",

  "recommended_action": "1-2 sentences stating exactly what a decision-maker 
    should do next. Be specific — avoid 'monitor closely' without naming what 
    to monitor and what threshold triggers escalation."
}

Return only valid JSON. No preamble, no explanation outside the JSON object."""


def interpret_kpi(metric_name, series_df, context):
    """
    Call the Claude API to generate a four-layer interpretation for one KPI.
    
    Args:
        metric_name: string name of the KPI
        series_df: DataFrame with 'date' and 'value' columns for this KPI
        context: string describing what this metric measures
    
    Returns:
        dict with four interpretation layers
    """
    # Format the time series as a clean text table for the prompt
    series_text = "\n".join([
        f"  {row['date'].strftime('%Y-%m-%d')}: {row['value']}"
        for _, row in series_df.iterrows()
    ])
    
    user_message = f"""Interpret this KPI time series.

Metric: {metric_name}
Context: {context}
Unit: {series_df['unit'].iloc[0] if 'unit' in series_df.columns else 'unknown'}

Weekly values (chronological):
{series_text}

Baseline (first 4 weeks avg): {series_df['value'].head(4).mean():.3f}
Recent (last 4 weeks avg): {series_df['value'].tail(4).mean():.3f}
Overall change: {((series_df['value'].tail(4).mean() - series_df['value'].head(4).mean()) / series_df['value'].head(4).mean() * 100):.1f}%
"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=800,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}]
    )
    
    raw = response.content[0].text.strip()
    
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()
    
    return json.loads(raw)


# ── Test on Zero Touch Rate first ────────────────────────────────────────────
ztr = kpi_data[kpi_data["metric_name"] == "Zero Touch Rate"].copy()

print("Sending Zero Touch Rate to Claude API...")
print(f"Series: {len(ztr)} weeks | Baseline avg: {ztr['value'].head(4).mean():.3f} | Recent avg: {ztr['value'].tail(4).mean():.3f}")
print()

result = interpret_kpi(
    metric_name="Zero Touch Rate",
    series_df=ztr,
    context=ztr["context"].iloc[0]
)

# Display the result
print("=" * 60)
print("INTERPRETATION: Zero Touch Rate")
print("=" * 60)
for layer, content in result.items():
    print(f"\n[{layer.upper()}]")
    print(content)

Sending Zero Touch Rate to Claude API...
Series: 12 weeks | Baseline avg: 0.087 | Recent avg: 0.224

INTERPRETATION: Zero Touch Rate

[WHAT_HAPPENED]
Zero Touch Rate rose steadily from 5.9% in the week of January 5 to 24.9% by March 23, 2026 — an increase of 19 percentage points over 12 consecutive weeks with no reversals. The baseline average (first 4 weeks) was 8.7%, while the most recent 4-week average stands at 22.4%, representing a 157.5% relative increase. The climb has been consistent at roughly 1.5–2 percentage points per week, suggesting a structural improvement rather than a one-time spike.

[SIGNAL_VS_NOISE]
The series shows no week-over-week declines and no high-frequency volatility — every single data point is higher than the one before it, across 12 weeks. A 157.5% increase from an 8.7% baseline with zero reversals is far beyond normal fluctuation for an automation rate metric. STRONG SIGNAL.

[CANDIDATE_EXPLANATIONS]
• Planned rollout of an automated resolution capabilit

In [4]:
# ============================================================================
# CELL 4: FULL INTERPRETATION LOOP — ALL THREE KPIs
# Run the interpretation function across the full dataset.
# Assess output quality on the spike (ticket volume) and dip (resolution rate)
# before prompt refinement.
# ============================================================================

def display_interpretation(metric_name, result):
    """Format and print a single KPI interpretation."""
    print("=" * 65)
    print(f"KPI: {metric_name}")
    print("=" * 65)
    for layer, content in result.items():
        label = layer.replace("_", " ").upper()
        print(f"\n[{label}]")
        print(content)
    print()


# Run all three KPIs
interpretations = {}

for metric_name in kpi_data["metric_name"].unique():
    series = kpi_data[kpi_data["metric_name"] == metric_name].copy()
    context = series["context"].iloc[0]
    
    print(f"Interpreting: {metric_name}...")
    interpretations[metric_name] = interpret_kpi(
        metric_name=metric_name,
        series_df=series,
        context=context
    )

print("\nAll interpretations complete.\n")
print("=" * 65)

# Display all results
for metric_name, result in interpretations.items():
    display_interpretation(metric_name, result)

Interpreting: Zero Touch Rate...
Interpreting: Weekly Ticket Volume...
Interpreting: Closed Within 1 Weekday Rate...

All interpretations complete.

KPI: Zero Touch Rate

[WHAT HAPPENED]
Zero Touch Rate has risen steadily and consistently from 5.9% in the week of January 5 to 24.9% by March 23, 2026 — a gain of 19 percentage points over 12 weeks with no reversals or plateaus. The early-period average (first 4 weeks) was 8.7%, while the most recent 4-week average stands at 22.4%, representing a 157.5% relative increase. The trajectory is smooth and nearly linear, suggesting a deliberate, sustained improvement rather than a one-time event.

[SIGNAL VS NOISE]
The change is 19 percentage points above baseline with zero week-over-week regressions across all 12 data points, indicating exceptionally low noise and a highly consistent directional trend. A 157.5% relative increase with this level of monotonic progression is far beyond normal operational variance. STRONG SIGNAL.

[CANDIDATE EXPLA

In [5]:
# ============================================================================
# CELL 5: OUTPUT FORMATTER
# Renders all interpretations as a clean, readable KPI report.
# Two formats:
#   (a) Terminal report — readable in Jupyter output
#   (b) Markdown string — exportable, GitHub-renderable
# ============================================================================

from datetime import datetime

def format_signal_badge(interpretation_text):
    """Extract and format the signal verdict from signal_vs_noise text."""
    text = interpretation_text.upper()
    if "STRONG SIGNAL" in text:
        return "🔴 STRONG SIGNAL"
    elif "MODERATE SIGNAL" in text:
        return "🟡 MODERATE SIGNAL"
    elif "NOISE" in text:
        return "🟢 NOISE"
    return "⚪ UNCLASSIFIED"


def render_terminal_report(interpretations, kpi_data):
    """Print a formatted terminal report for all KPIs."""
    
    report_date = datetime.today().strftime("%B %d, %Y")
    
    print("=" * 70)
    print("  KPI INTERPRETATION REPORT")
    print(f"  Generated: {report_date}")
    print(f"  KPIs analyzed: {len(interpretations)}")
    print(f"  Period: {kpi_data['date'].min().strftime('%b %d')} – "
          f"{kpi_data['date'].max().strftime('%b %d, %Y')}")
    print("=" * 70)
    
    for metric_name, result in interpretations.items():
        signal_badge = format_signal_badge(result['signal_vs_noise'])
        
        # Header
        print(f"\n{'─' * 70}")
        print(f"  {metric_name}")
        print(f"  {signal_badge}")
        print(f"{'─' * 70}")
        
        # Four layers
        layer_labels = {
            "what_happened":        "WHAT HAPPENED",
            "signal_vs_noise":      "SIGNAL VS NOISE",
            "candidate_explanations": "CANDIDATE EXPLANATIONS",
            "recommended_action":   "RECOMMENDED ACTION"
        }
        
        for key, label in layer_labels.items():
            content = result.get(key, "")
            print(f"\n  [{label}]")
            # Word-wrap at 65 chars for readability
            words = content.split()
            line = "  "
            for word in words:
                if len(line) + len(word) + 1 > 67:
                    print(line)
                    line = "  " + word + " "
                else:
                    line += word + " "
            if line.strip():
                print(line)
    
    print(f"\n{'=' * 70}")
    print("  END OF REPORT")
    print("=" * 70)


def render_markdown_report(interpretations, kpi_data):
    """Generate a markdown string — exportable and GitHub-renderable."""
    
    report_date = datetime.today().strftime("%B %d, %Y")
    lines = []
    
    lines.append("# KPI Interpretation Report")
    lines.append(f"**Generated:** {report_date}  ")
    lines.append(f"**Period:** {kpi_data['date'].min().strftime('%b %d')} – "
                 f"{kpi_data['date'].max().strftime('%b %d, %Y')}  ")
    lines.append(f"**KPIs analyzed:** {len(interpretations)}")
    lines.append("")
    lines.append("---")
    
    for metric_name, result in interpretations.items():
        signal_badge = format_signal_badge(result['signal_vs_noise'])
        
        lines.append(f"\n## {metric_name}")
        lines.append(f"**{signal_badge}**")
        lines.append("")
        
        lines.append("### What Happened")
        lines.append(result.get("what_happened", ""))
        lines.append("")
        
        lines.append("### Signal vs. Noise")
        lines.append(result.get("signal_vs_noise", ""))
        lines.append("")
        
        lines.append("### Candidate Explanations")
        # Preserve bullet formatting from the model
        explanations = result.get("candidate_explanations", "")
        if "•" in explanations:
            for bullet in explanations.split("•"):
                bullet = bullet.strip()
                if bullet:
                    lines.append(f"- {bullet}")
        else:
            lines.append(explanations)
        lines.append("")
        
        lines.append("### Recommended Action")
        lines.append(result.get("recommended_action", ""))
        lines.append("")
        lines.append("---")
    
    return "\n".join(lines)


# ── Run both formatters ───────────────────────────────────────────────────────
render_terminal_report(interpretations, kpi_data)

print("\n\n")
print("MARKDOWN VERSION (copy-paste or export):")
print("=" * 70)
markdown_output = render_markdown_report(interpretations, kpi_data)
print(markdown_output)

  KPI INTERPRETATION REPORT
  Generated: May 05, 2026
  KPIs analyzed: 3
  Period: Jan 05 – Mar 23, 2026

──────────────────────────────────────────────────────────────────────
  Zero Touch Rate
  🔴 STRONG SIGNAL
──────────────────────────────────────────────────────────────────────

  [WHAT HAPPENED]
  Zero Touch Rate has risen steadily and consistently from 5.9% in 
  the week of January 5 to 24.9% by March 23, 2026 — a gain of 19 
  percentage points over 12 weeks with no reversals or plateaus. 
  The early-period average (first 4 weeks) was 8.7%, while the 
  most recent 4-week average stands at 22.4%, representing a 
  157.5% relative increase. The trajectory is smooth and nearly 
  linear, suggesting a deliberate, sustained improvement rather 
  than a one-time event. 

  [SIGNAL VS NOISE]
  The change is 19 percentage points above baseline with zero 
  week-over-week regressions across all 12 data points, indicating 
  exceptionally low noise and a highly consistent directional 

In [6]:
# ============================================================================
# CELL 6: BULLET FIX + MARKDOWN EXPORT + INTERVIEW NARRATIVE
# ============================================================================

# ── Fix 1: Updated terminal formatter with proper bullet separation ───────────

def render_terminal_report_v2(interpretations, kpi_data):
    """Terminal report with corrected bullet point line separation."""
    
    report_date = datetime.today().strftime("%B %d, %Y")
    
    print("=" * 70)
    print("  KPI INTERPRETATION REPORT")
    print(f"  Generated: {report_date}")
    print(f"  KPIs analyzed: {len(interpretations)}")
    print(f"  Period: {kpi_data['date'].min().strftime('%b %d')} – "
          f"{kpi_data['date'].max().strftime('%b %d, %Y')}")
    print("=" * 70)
    
    for metric_name, result in interpretations.items():
        signal_badge = format_signal_badge(result['signal_vs_noise'])
        
        print(f"\n{'─' * 70}")
        print(f"  {metric_name}")
        print(f"  {signal_badge}")
        print(f"{'─' * 70}")
        
        layer_labels = {
            "what_happened":          "WHAT HAPPENED",
            "signal_vs_noise":        "SIGNAL VS NOISE",
            "candidate_explanations": "CANDIDATE EXPLANATIONS",
            "recommended_action":     "RECOMMENDED ACTION"
        }
        
        for key, label in layer_labels.items():
            content = result.get(key, "")
            print(f"\n  [{label}]")
            
            if key == "candidate_explanations" and "•" in content:
                # Split on bullet character and print each on its own line
                bullets = [b.strip() for b in content.split("•") if b.strip()]
                for bullet in bullets:
                    # Word-wrap each bullet at 63 chars
                    words = bullet.split()
                    line = "  • "
                    for word in words:
                        if len(line) + len(word) + 1 > 67:
                            print(line)
                            line = "    " + word + " "
                        else:
                            line += word + " "
                    if line.strip():
                        print(line)
                    print()  # blank line between bullets
            else:
                # Standard word-wrap for other layers
                words = content.split()
                line = "  "
                for word in words:
                    if len(line) + len(word) + 1 > 67:
                        print(line)
                        line = "  " + word + " "
                    else:
                        line += word + " "
                if line.strip():
                    print(line)
    
    print(f"\n{'=' * 70}")
    print("  END OF REPORT")
    print("=" * 70)


# ── Fix 2: Markdown export to file ───────────────────────────────────────────

def export_markdown_report(interpretations, kpi_data, filename="kpi_report.md"):
    """Save the markdown report to a file."""
    markdown = render_markdown_report(interpretations, kpi_data)
    with open(filename, "w") as f:
        f.write(markdown)
    print(f"Markdown report saved to: {filename}")
    return filename


# ── Run the corrected terminal report ────────────────────────────────────────
render_terminal_report_v2(interpretations, kpi_data)

# ── Export markdown to file ───────────────────────────────────────────────────
export_markdown_report(interpretations, kpi_data, "kpi_report.md")

  KPI INTERPRETATION REPORT
  Generated: May 05, 2026
  KPIs analyzed: 3
  Period: Jan 05 – Mar 23, 2026

──────────────────────────────────────────────────────────────────────
  Zero Touch Rate
  🔴 STRONG SIGNAL
──────────────────────────────────────────────────────────────────────

  [WHAT HAPPENED]
  Zero Touch Rate has risen steadily and consistently from 5.9% in 
  the week of January 5 to 24.9% by March 23, 2026 — a gain of 19 
  percentage points over 12 weeks with no reversals or plateaus. 
  The early-period average (first 4 weeks) was 8.7%, while the 
  most recent 4-week average stands at 22.4%, representing a 
  157.5% relative increase. The trajectory is smooth and nearly 
  linear, suggesting a deliberate, sustained improvement rather 
  than a one-time event. 

  [SIGNAL VS NOISE]
  The change is 19 percentage points above baseline with zero 
  week-over-week regressions across all 12 data points, indicating 
  exceptionally low noise and a highly consistent directional 

'kpi_report.md'

---

## What This Tool Demonstrates

This notebook implements a **KPI Interpretation Tool** using the Claude API
(`claude-sonnet-4-6`). It takes a metric time series and returns a structured,
four-layer natural language interpretation designed for senior operations leaders.

### The Four-Layer Framework

| Layer | Question | Output |
|---|---|---|
| What happened | Trend description | Direction, magnitude, duration — specific numbers |
| Signal vs. noise | Is this meaningful? | Verdict: STRONG SIGNAL / MODERATE SIGNAL / NOISE |
| Candidate explanations | Why might this be happening? | Ordered plausible mechanisms |
| Recommended action | What should we do? | Specific next step with named threshold |

This framework maps directly to a measurement philosophy I've applied throughout
my career: analytics should answer four questions — what happened, why it
happened, what's likely next, and what we should do about it. A report that
stops at "the metric went up" hasn't finished the job.

### What this demonstrates technically

- **Claude API integration** via the Anthropic Python SDK — not just the chat
  interface, but programmatic API access with a structured system prompt
- **Prompt engineering for structured output** — the system prompt enforces
  valid JSON with four specific fields, and the user message pre-computes
  summary statistics to improve analytical quality
- **Signal classification** — the tool distinguishes between strong trends,
  discrete anomalies, and noise, which requires the prompt to encode
  analytical judgment, not just summarization instructions
- **Output formatting** — terminal and markdown renderers with signal badges,
  exportable report file

### Design decisions worth noting

The prompt pre-computes baseline average, recent average, and percentage change
before passing data to the model. This reduces the model's arithmetic burden and
produces more consistent quantitative references in the output. The signal
verdict (STRONG / MODERATE / NOISE) is enforced in the system prompt rather than
inferred post-hoc — which means the model commits to a classification as part of
its reasoning, not as a label applied afterward.

The dataset uses IT service operations KPIs from the same domain as my
quasi-experimental chatbot impact study and bootstrapped matching analysis.
The tool is designed to automate the first layer of the interpretive work
that currently requires an analyst to read and contextualize each metric
before surfacing findings to service managers.